# SAP AI Core — Material Consumption Prediction
## Setup Notebook
This notebook walks through all the setup steps:
1. Install dependencies
2. Create Resource Group
3. Register Git repository (with .aicore/ templates)
4. Register Object Store secret
5. Register Docker Registry secret
6. Upload training data
7. Register Artifact
8. Create Configuration
9. Run Training Execution
10. Deploy Serving
11. Test Prediction

In [1]:
!pip install requests boto3 --quiet
print('Dependencies ready')

Dependencies ready


## 1. Configure credentials
Fill in the values from your AI Core service key in BTP.

In [ ]:
import requests, base64, json, time, os

AI_CORE_BASE_URL     = 'https://api.ai.prod.us-east-1.aws.ml.hana.ondemand.com/v2'
AI_CORE_TOKEN_URL    = 'https://i516195-dev-2bqwzxca.authentication.us10.hana.ondemand.com/oauth/token'
AI_CORE_CLIENT_ID    = os.environ['AICORE_CLIENT_ID']
AI_CORE_CLIENT_SECRET= os.environ['AICORE_CLIENT_SECRET']
RESOURCE_GROUP       = 'consumption-rg'
SCENARIO_ID          = 'material-consumption'

def get_token():
    resp = requests.post(
        AI_CORE_TOKEN_URL,
        data={'grant_type': 'client_credentials'},
        auth=(AI_CORE_CLIENT_ID, AI_CORE_CLIENT_SECRET),
    )
    resp.raise_for_status()
    return resp.json()['access_token']

def headers(rg=None):
    h = {'Authorization': f'Bearer {get_token()}', 'Content-Type': 'application/json'}
    if rg:
        h['AI-Resource-Group'] = rg
    return h

TOKEN = get_token()
print('Connected to AI Core ✓')

## 2. Create Resource Group

In [3]:
resp = requests.post(
    f'{AI_CORE_BASE_URL}/admin/resourceGroups',
    headers=headers(),
    json={'resourceGroupId': RESOURCE_GROUP},
)
print(f'Resource group: {resp.status_code} — {resp.json()}')

Resource group: 409 — {'error': {'code': '05060002', 'details': {}, 'message': 'Resource group with tenantId 59799258-5e71-4a68-ac79-02f7d5254efd and resource group id consumption-rg already exists.', 'status': 409, 'target': '/api/v1/resourcegroups'}}


## 3. Register Git Repository
> Push the project folder (containing `.aicore/`) to a Git repository first.
> Then register it below.

In [ ]:
resp = requests.post(
    f'{AI_CORE_BASE_URL}/admin/applications',
    headers=headers(),
    json={
        'applicationName': 'consumption-app',
        'repositoryUrl':   'https://github.com/cguevara962/AI_Core_prueba',
        'revision':        'HEAD',
        'path':            '.aicore',
    },
)
print(f'Git repo: {resp.status_code} — {resp.json()}')

## 4. Register Object Store Secret
SAP AI Core uses Object Store to read training data and write model artifacts.
Supported: AWS S3, Azure Blob, SAP BTP Object Store Service.

In [ ]:
import os

BUCKET     = os.environ['S3_BUCKET']
ENDPOINT   = os.environ['S3_ENDPOINT']
REGION     = os.environ['S3_REGION']
ACCESS_KEY = os.environ['S3_ACCESS_KEY']
SECRET_KEY = os.environ['S3_SECRET_KEY']

resp = requests.post(
    f'{AI_CORE_BASE_URL}/admin/objectStoreSecrets',
    headers=headers(),
    json={
        'name':       'default',
        'type':       'S3',
        'endpoint':   ENDPOINT,
        'bucket':     BUCKET,
        'pathPrefix': 'consumption-ai/',
        'region':     REGION,
        'data': {
            'AWS_ACCESS_KEY_ID':     ACCESS_KEY,
            'AWS_SECRET_ACCESS_KEY': SECRET_KEY,
        },
    },
)
print(f'Object store secret: {resp.status_code} — {resp.json()}')

## 5. Register Docker Registry Secret

In [ ]:
DOCKERHUB_USER  = 'cguevara962'
DOCKERHUB_TOKEN = os.environ['DOCKERHUB_TOKEN']

docker_config = {
    "auths": {
        "https://index.docker.io/v1/": {
            "auth": base64.b64encode(
                f"{DOCKERHUB_USER}:{DOCKERHUB_TOKEN}".encode()
            ).decode()
        }
    }
}

resp = requests.post(
    f'{AI_CORE_BASE_URL}/admin/dockerRegistrySecrets',
    headers=headers(),
    json={
        'name': 'docker-registry-secret',
        'data': {'.dockerconfigjson': json.dumps(docker_config)},
    },
)
print(f'Docker secret: {resp.status_code} — {resp.json()}')

## 6. Register Training Data Artifact
Upload `data/consumption.csv` to your Object Store bucket first,
then register it as an artifact.

In [7]:
import boto3

s3 = boto3.client(
    's3',
    endpoint_url          = f'https://{ENDPOINT}',
    aws_access_key_id     = ACCESS_KEY,
    aws_secret_access_key = SECRET_KEY,
    region_name           = REGION,
)

csv_local_path = '../cap-app/db/data/consumption-ConsumptionHistory.csv'
s3_key         = 'consumption-ai/data/consumption.csv'

s3.upload_file(csv_local_path, BUCKET, s3_key)
print(f'Uploaded → s3://{BUCKET}/{s3_key} ✓')

Uploaded → s3://hcp-c096a718-bfa7-4194-858b-01b0ed9a3609/consumption-ai/data/consumption.csv ✓


In [9]:
resp = requests.post(
    f'{AI_CORE_BASE_URL}/lm/artifacts',
    headers=headers(RESOURCE_GROUP),
    json={
        'name':        'historical-consumption-data',
        'kind':        'dataset',
        'url':         'ai://default/consumption-ai/data',
        'scenarioId':  SCENARIO_ID,
        'description': '2 years of daily material consumption history',
    },
)
resp.raise_for_status()
ARTIFACT_ID = resp.json()['id']
print(f'Artifact ID: {ARTIFACT_ID}')

HTTPError: 400 Client Error: Bad Request for url: https://api.ai.prod.us-east-1.aws.ml.hana.ondemand.com/v2/lm/artifacts

## 7. Create Training Configuration

In [ ]:
resp = requests.post(
    f'{AI_CORE_BASE_URL}/lm/configurations',
    headers=headers(RESOURCE_GROUP),
    json={
        'name':         'consumption-training-config-v1',
        'scenarioId':   SCENARIO_ID,
        'executableId': 'consumption-training',
        'parameterBindings': [
            {'key': 'n_estimators', 'value': '200'},
            {'key': 'max_depth',    'value': '8'},
            {'key': 'test_size',    'value': '0.2'},
        ],
        'inputArtifactBindings': [
            {'key': 'historical-data', 'artifactId': ARTIFACT_ID},
        ],
    },
)
resp.raise_for_status()
TRAINING_CONFIG_ID = resp.json()['id']
print(f'Training config ID: {TRAINING_CONFIG_ID}')

## 8. Run Training Execution

In [ ]:
resp = requests.post(
    f'{AI_CORE_BASE_URL}/lm/executions',
    headers=headers(RESOURCE_GROUP),
    json={'configurationId': TRAINING_CONFIG_ID},
)
resp.raise_for_status()
EXECUTION_ID = resp.json()['id']
print(f'Execution ID: {EXECUTION_ID}  |  Status: {resp.json().get("status")}')

In [ ]:
# Poll until COMPLETED (training can take 5-15 min)
while True:
    resp = requests.get(
        f'{AI_CORE_BASE_URL}/lm/executions/{EXECUTION_ID}',
        headers=headers(RESOURCE_GROUP),
    )
    status = resp.json().get('status')
    print(f'Status: {status}')
    if status in ['COMPLETED', 'DEAD', 'STOPPED']:
        break
    time.sleep(30)

# Get trained-model artifact ID
resp = requests.get(
    f'{AI_CORE_BASE_URL}/lm/artifacts',
    headers=headers(RESOURCE_GROUP),
    params={'scenarioId': SCENARIO_ID, 'kind': 'model'},
)
artifacts = resp.json().get('resources', [])
model_artifact = next(a for a in artifacts if a['name'] == 'trained-model')
MODEL_ARTIFACT_ID = model_artifact['id']
print(f'Model artifact ID: {MODEL_ARTIFACT_ID}')

## 9. Create Serving Configuration & Deploy

In [ ]:
resp = requests.post(
    f'{AI_CORE_BASE_URL}/lm/configurations',
    headers=headers(RESOURCE_GROUP),
    json={
        'name':         'consumption-serving-config-v1',
        'scenarioId':   SCENARIO_ID,
        'executableId': 'consumption-serving',
        'inputArtifactBindings': [
            {'key': 'trained-model', 'artifactId': MODEL_ARTIFACT_ID},
        ],
    },
)
resp.raise_for_status()
SERVE_CONFIG_ID = resp.json()['id']

resp = requests.post(
    f'{AI_CORE_BASE_URL}/lm/deployments',
    headers=headers(RESOURCE_GROUP),
    json={'configurationId': SERVE_CONFIG_ID},
)
resp.raise_for_status()
DEPLOYMENT_ID = resp.json()['id']
print(f'Deployment ID: {DEPLOYMENT_ID}')

In [ ]:
# Poll until RUNNING (can take 5-10 min)
while True:
    resp = requests.get(
        f'{AI_CORE_BASE_URL}/lm/deployments/{DEPLOYMENT_ID}',
        headers=headers(RESOURCE_GROUP),
    )
    dep = resp.json()
    status = dep.get('status')
    print(f'Deployment status: {status}')
    if status == 'RUNNING':
        DEPLOYMENT_URL = dep['deploymentUrl']
        print(f'\nDeployment URL: {DEPLOYMENT_URL}')
        print('\n→ Copia esta URL en cap-app/.env como AICORE_DEPLOYMENT_URL')
        break
    if status in ['DEAD', 'STOPPED']:
        print('Deployment failed:', dep)
        break
    time.sleep(30)

## 10. Test Prediction

In [ ]:
token_resp = requests.post(
    AI_CORE_TOKEN_URL,
    data={'grant_type': 'client_credentials'},
    auth=(AI_CORE_CLIENT_ID, AI_CORE_CLIENT_SECRET),
)
token = token_resp.json()['access_token']

payload = {
    'inputs': [{'data': [
        {'material_id':'MAT-001','date':'2026-08-04',
         'is_holiday':False,'is_weekend':False,'is_payday':True,
         'lag_7d':195.4,'lag_14d':188.1,'lag_28d':201.6,'rolling_4w_avg':193.2},
        {'material_id':'MAT-002','date':'2026-08-04',
         'is_holiday':False,'is_weekend':False,'is_payday':True,
         'lag_7d':78.0,'lag_14d':82.5,'lag_28d':79.0,'rolling_4w_avg':80.1},
    ]}]
}

resp = requests.post(
    f'{DEPLOYMENT_URL}/v2/models/consumption-model/infer',
    json=payload,
    headers={
        'Authorization':    f'Bearer {token}',
        'AI-Resource-Group': RESOURCE_GROUP,
    },
)
print(resp.json())

## Done!
Copy `DEPLOYMENT_URL` to your CAP app's `.env` file as `AICORE_DEPLOYMENT_URL`,
then run `npm run dev` inside `cap-app/` to start the application.
Call the `refreshPredictions` action to populate today's predictions.